# Fixed data center: minimum-cost HPP sizing
Use hydesign_clean and Run All. Initial case: fixed 16 MW facility load, 99.9% hourly
reliability, wind turbine count only, PV 180 MW AC and battery 25 MW / 200 MWh.
The reference design is checked with reliability enforced; feasibility is never assumed.

Select feasibility_aware (default) or native_ego in the configuration below. Both
methods reuse the existing thesis EMS, physical validation and cost equations. Results are
best validated feasible designs found, not certified global optima. The 600 s outer
budget is best effort: construction, surrogate work, active dispatch solves and final
validation can overrun. No new candidates start after the deadline. Each CPLEX solve
is limited separately by ems_time_limit_seconds and remaining outer time.

The EMS uses one CPLEX dispatch solve. Set battery_power_fluctuation_penalty
in YAML (default 0.0; optional trials 0.0001 or 0.001).
Its units are MWh_IT per MW of adjacent change in AC net battery power
(discharge - charge); there is no end-to-start ramp or throughput penalty.
Fixed/basic dispatch maximizes served IT MWh minus this energy-equivalent penalty.
Economic sizing multiplies the penalty by the IT revenue rate to obtain EUR.
Economic reports exclude the artificial penalty; solver objectives/bounds include it.

The unchanged validator rejects simultaneous charging and discharging. The existing
physical formulation permits this, and neither zero nor a positive ramp penalty
guarantees its elimination. The annual reference evaluation can therefore stop with
"Simultaneous battery charging/discharging"; do not use rejected dispatch as a result.

Shared infrastructure is always included: G_proxy_MW = facility_load_mw + rated b_P,
without efficiency correction. Native shared_cost uses BOS 119,940 EUR/MW and
land 300,000 EUR/km2 times max(Awpp, Apvp); public-grid cost is zero.
This capacity proxy is not a detailed electrical-network design. Shared CAPEX
is annualized using the HPP rates (6%) and economic lifetime (25 years).

Battery economics include one full replacement in year 15, discounted at the
battery WACC (6%) and annualized across the HPP economic lifetime (25 years).
CAPEX_b and CAPEX_hpp remain year-0 investment only. The new replacement fields
are expenditures in EUR, not EUR/year. There is no physical degradation, price
reduction or salvage value; hourly dispatch and reliability are unchanged.


In [ ]:
import os
os.environ['OPENMDAO_REPORTS'] = '0'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hydesign
from hydesign.examples import examples_filepath
from hydesign.sizing_simple_datacenter_with_hpp_sizing import read_configuration, run_hpp_sizing, VARIABLE_NAMES

repo = Path(hydesign.__file__).resolve().parent.parent
print('Python:', sys.executable)
print('Repository:', repo)
config_path = Path(examples_filepath) / 'thesis_simple_datacenter/simple_datacenter.yml'
config = read_configuration(config_path)


## Configurable scenario
Edit values here or in the new YAML. Switch var_type between 'fixed' and 'design';
value and limits remain configurable. Case A is wind only; case B sets Nwt fixed and
PV/battery variables design; case C sets all four design. All fixed evaluates once.
Higher reliability may be infeasible; no constraint relaxation is performed.


In [ ]:
# Defaults are read from YAML. Uncomment a line to override this run only.
# config['optimization_method'] = 'native_ego' # Or 'feasibility_aware'
config['facility_load_mw'] = 16
config['reliability_target'] = 0.99
config['hpp_sizing_time_limit_seconds'] = 60
config['hpp_sizing_output_dir'] = str(repo / 'hpp_sizing_runs')
variables = config['variables']
# YAML initially selects case A. Uncomment/edit flags for another scenario.
variables['Nwt']['var_type'] = 'design'
# variables['Nwt']['value'] = 20
# variables['solar_MW [MW AC]']['var_type'] = 'design'
# variables['b_P [MW]']['var_type'] = 'fixed'
# variables['b_E_h [h]']['var_type'] = 'fixed'
# Optional fixed values / bounds / search overrides:
# variables['solar_MW [MW AC]']['value'] = 180
# variables['Nwt']['limits'] = [0, 400]
# config['n_doe'] = 16
# config['fa_batch_size'] = 2
# config['ems_time_limit_seconds'] = 600
print('Outer budget / inner solve limit [s]:', config['hpp_sizing_time_limit_seconds'], config['ems_time_limit_seconds'])
print('Facility MW:', config['facility_load_mw'])
print('IT MW:', config['facility_load_mw'] / config['pue'])
print('Reliability:', config['reliability_target'])
print('Allowed failed hours:', int(np.floor((1-config['reliability_target'])*8760 + 1e-9)))
display(pd.DataFrame(variables).T)
print('Method:', config['optimization_method'])
print('Search settings:', {k: config[k] for k in ['n_doe','max_iter','n_clusters','npred','n_procs','n_seed','tol','min_conv_iter']})
if config['optimization_method'] == 'feasibility_aware':
    print('Feasibility-aware settings:', {k: v for k, v in config.items() if k.startswith('fa_')})


In [ ]:
result = run_hpp_sizing(config_path, overrides=config)
print('Results:', result['run_directory'])


In [ ]:
print('Method:', result['optimization_method'])
print('Termination:', result['termination_reason'])
print('Error:', result['error'])
print('Elapsed seconds:', result['elapsed_seconds'])
print('Evaluations / unique designs / feasible:', result['evaluated_designs'], result['unique_designs'], result['feasible_designs'])
print('Search diagnostics:', result['search_summary'])
print('Site:', result['configuration']['hpp_sizing_site'])
print('Objective: minimize annualized_cost_hpp; no global optimality certificate')
print('Bound hits:', result['bound_hits'])
best = result['best']
if best is None:
    print('No validated feasible design found. This does not prove the entire search domain infeasible.')
else:
    names = VARIABLE_NAMES
    display(pd.Series(dict(zip(names, best['design'])), name='Best feasible design'))
    print('Wind MW:', best['design'][3] * best['design'][2])
    print('Battery MWh:', best['design'][-2] * best['design'][-1])
    print('Allowed failed hours:', result['final_validation']['max_failed_hours'])
    display(pd.Series(best['metrics'], name='Technical results'))
    display(pd.Series(best['economics'], name='Economics (initial/replacement costs EUR; annual values EUR/year)'))
    print('Solver status:', best['solver_status'])
    print('Independent validation:', result['final_validation'])


In [ ]:
history_path = Path(result['history_path'])
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history)
    history.plot(x='elapsed_seconds', y='best_feasible_cost', marker='o', legend=False)
    plt.ylabel('Best validated annual HPP cost [EUR/year]')
    plt.xlabel('Elapsed time [s]')
    plt.show()
else:
    print('No candidate completed before termination.')
print('Predictions recorded before evaluations:', result['proposals_path'])


## Interpretation
Reliability remains a hard hourly EMS constraint; EENS is reported separately.
The feasibility-aware method fits native KPLSK to genuine accepted HPP costs and a
separate Gaussian-process classifier to validated feasible/proven-infeasible labels.
It ranks global mixed-integer samples using cost expected improvement times estimated
feasibility probability. Periodic global exploration also samples uncertain regions.
This probability is a search prediction, not the physical reliability of the plant.

Unknown solver outcomes are excluded from both training datasets and may receive a
bounded retry. Sparse or one-class datasets use documented exploration/prior fallbacks.
There is no local or neighbour search. Native EGO retains its original scalar adapter
scores, including rejection scores; the feasibility-aware cost model never uses them.

The saved best design and result JSON are authoritative. Neither search certifies a
global minimum. Timeouts remain unaccepted until both unchanged EMS solves and physical
validation complete. Shared BOS and co-located land costs are included. Public-grid connection,
decommissioning and degradation remain excluded. One year-15 battery replacement is included. DC economics remain reporting quantities. Runs use unique directories;
the parent process writes completed evaluations, proposal predictions and incumbents.
Do not launch concurrent searches from the same Python process.
